# Gnani Prisma (ASR) & Timbre (TTS) Evaluation

**Engineer:** Ashish Vipin  
**Target Systems:** Gnani Prisma v3.0 (ASR) & Timbre v2.5 (TTS) via `api.vachana.ai`

---

## Overview
1. **Part 1 — Prisma ASR**: Evaluates **Clean Real-Human Speech** (18 clips: Svarah `CC BY 4.0`, FLEURS `CC BY 4.0`, OpenSLR 66/65/37 `CC BY-SA 4.0`), **Paired Acoustic Stress** (16 clips across 20/10/5 dB babble & 8 kHz μ-law telephony), **Synthetic Smoke** (30 clips), and a **Hinglish ASR Language Ablation** (`hi-IN` vs `hi-IN,en-IN` vs `indic-auto`). Primary scoring uses Verbatim WER/CER plus dataset-agnostic `normalise_for_eval` rules and 95% clip-level bootstrap confidence intervals.
2. **Part 2 — Timbre TTS**: Evaluates 29 sentences × 3 REST repeats (`/api/v1/tts/inference`) and 5 sentences × 3 SSE streaming repeats (`/api/v1/tts/sse`) via component metrics—streaming TTFB, REST latency/RTF, 25 ms frame RMS silence ratio, clipping, and round-trip Prisma WER against raw input `text`.

In [1]:
import sys
import json
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))

from config import GNANI_API_KEY, ASR_MODEL, TTS_MODEL
from src.metrics import compute_wer, compute_rule_wer, normalise_for_eval
from src.gnani_client import GnaniClient

print('API Key Configured:', 'Yes' if GNANI_API_KEY else 'No')
print(f'Target Models: ASR={ASR_MODEL}, TTS={TTS_MODEL}')

API Key Configured: Yes
Target Models: ASR=prisma-v3.0, TTS=timbre-v2.5


/Users/mac/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## Optional: Re-Run Live Evaluation Directly From This Notebook
Set `RUN_LIVE_EVAL = True` to re-run all ASR suites + TTS REST & SSE streaming benchmarks live against `api.vachana.ai`.

In [2]:
RUN_LIVE_EVAL = False
TTS_REPEATS = 3
USE_INDEPENDENT_WHISPER = False  # Set True on Colab (Python >= 3.10) after `!pip install faster-whisper`

if RUN_LIVE_EVAL:
    from src.asr_evaluator import run_asr_suites
    from src.tts_evaluator import run_tts_evaluation

    run_asr_suites(
        suites={
            'real_human': 'data/asr/real_manifest.json',
            'real_paired_stress': 'data/asr/real_stress_manifest.json',
            'synthetic_smoke': 'data/asr/test_manifest.json',
        },
        output_path='results/asr_results.json',
        include_hinglish_ablation=True,
    )
    run_tts_evaluation(
        test_set_path='data/tts/test_sentences.json',
        output_dir='data/tts/output',
        output_path='results/tts_results.json',
        repeats=TTS_REPEATS,
        run_streaming_benchmark=True,
        use_independent_asr=USE_INDEPENDENT_WHISPER,
    )
else:
    print('RUN_LIVE_EVAL=False — loading saved empirical runs from results/*.json')

RUN_LIVE_EVAL=False — loading saved empirical runs from results/*.json


## Part 1: Scratch DP Metrics & Dataset-Agnostic Rule Normaliser

In [3]:
sample_ref = 'from eleventh to fourteenth century CE'
sample_hyp = 'from 11th to 14th century CE'
res_verb = compute_wer(sample_ref, sample_hyp)
res_rule = compute_rule_wer(sample_ref, sample_hyp)
print('Normalised rule string:', repr(normalise_for_eval(sample_ref)))
print(f"Verbatim WER={res_verb['wer']:.2%} | Rule-Normalised WER={res_rule['wer']:.2%}")

Normalised rule string: 'from 11 to 14 century ce'
Verbatim WER=33.33% | Rule-Normalised WER=0.00%


## Part 1 Results: Prisma ASR (Real Clean, Paired Stress, Synthetic Smoke & Hinglish Ablation)

In [4]:
with open('results/asr_results.json', 'r', encoding='utf-8') as f:
    asr_res = json.load(f)

for suite_name, suite in asr_res['suites'].items():
    ov = suite['overall']
    print('=' * 78)
    print(f"SUITE: {suite_name} (n={suite['metadata']['successful_clips']}/{suite['metadata']['total_clips']})")
    print(f"Verbatim WER: {ov['wer']:.2%} [95% CI: {ov['wer_ci_95'][0]:.2%}–{ov['wer_ci_95'][1]:.2%}]")
    print(f"Rule-Norm WER: {ov['rule_normalised_wer']:.2%} [95% CI: {ov['rule_wer_ci_95'][0]:.2%}–{ov['rule_wer_ci_95'][1]:.2%}]")
    print(f"Latency: p50={ov['p50_latency_ms']:.0f}ms, p95={ov['p95_latency_ms']:.0f}ms, cross_script_subs={ov['cross_script_subs']}, retries={ov['total_retries']}")
    print('-' * 78)
    print(f"{'Category':<24} | {'Verb WER':>8} | {'Rule WER':>8} | {'p50 Lat':>8} | {'Clips':>5}")
    for cat, d in suite['by_category'].items():
        print(f"{cat:<24} | {d['aggregate_wer']*100:>7.2f}% | {d['rule_normalised_wer']*100:>7.2f}% | {d['p50_latency_ms']:>6.0f}ms | {d['num_clips']:>5}")
    print()

print('Hinglish Language-Parameter Ablation:')
for lang_opt, d in asr_res.get('hinglish_ablation', {}).items():
    print(f"  language={lang_opt:<11s} -> Verbatim WER={d['verbatim_wer']:.2%}, p50={d['p50_latency_ms']:.0f}ms")

SUITE: real_human (n=18/18)
Verbatim WER: 10.93% [95% CI: 5.85%–16.09%]
Rule-Norm WER: 10.93% [95% CI: 5.85%–16.09%]
Latency: p50=434ms, p95=656ms, cross_script_subs=14, retries=0
------------------------------------------------------------------------------
Category                 | Verb WER | Rule WER |  p50 Lat | Clips
real_indian_english      |    1.45% |    1.45% |    440ms |     5
real_hindi               |   14.63% |   14.63% |    515ms |     5
real_regional_indic      |   14.55% |   14.55% |    360ms |     8

SUITE: real_paired_stress (n=16/16)
Verbatim WER: 5.92% [95% CI: 3.42%–8.60%]
Rule-Norm WER: 5.92% [95% CI: 3.42%–8.60%]
Latency: p50=424ms, p95=583ms, cross_script_subs=5, retries=0
------------------------------------------------------------------------------
Category                 | Verb WER | Rule WER |  p50 Lat | Clips
paired_babble_20db       |    3.95% |    3.95% |    487ms |     4
paired_babble_10db       |    6.58% |    6.58% |    523ms |     4
paired_babble_5d

## Part 2 Results: Timbre TTS (REST Component Metrics + SSE Streaming TTFB)

In [5]:
with open('results/tts_results.json', 'r', encoding='utf-8') as f:
    tts_res = json.load(f)

ov = tts_res['overall']
meta = tts_res['metadata']
sse = tts_res.get('streaming_ttfb_benchmark', {})
print('=' * 78)
print(f"TIMBRE TTS BENCHMARK (n={meta['successful']}/{meta['total_sentences']}, repeats={meta['repeats_per_sentence']})")
print(f"REST HTTP Latency: p50={ov['p50_latency_ms']:.0f}ms, p95={ov['p95_latency_ms']:.0f}ms, retries={ov['total_retries']}")
if sse:
    print(f"SSE Streaming TTFB: p50={sse['p50_ttfb_first_chunk_ms']:.0f}ms, p95={sse['p95_ttfb_first_chunk_ms']:.0f}ms (total stream p50={sse['p50_total_stream_ms']:.0f}ms)")
print(f"RTF: p50={ov['p50_rtf']:.4f}, p95={ov['p95_rtf']:.4f}  |  25ms Silence Ratio: {ov['avg_silence_ratio']:.3f}")
print(f"Round-trip Prisma WER (vs raw text): verbatim={ov['avg_roundtrip_verbatim_wer']:.2%}, rule-normalised={ov['avg_roundtrip_rule_wer']:.2%}")
print('=' * 78)
print(f"{'Category':<24} | {'p50 Lat':>7} | {'p95 Lat':>7} | {'p50 RTF':>7} | {'Sil':>5} | {'RT RuleWER':>10}")
for cat, d in tts_res['by_category'].items():
    rt = f"{d['avg_roundtrip_rule_wer']*100:.1f}%" if d['avg_roundtrip_rule_wer'] is not None else 'n/a'
    print(f"{cat:<24} | {d['p50_latency_ms']:>5.0f}ms | {d['p95_latency_ms']:>5.0f}ms | {d['p50_rtf']:>7.4f} | {d['avg_silence_ratio']:>5.3f} | {rt:>10}")

TIMBRE TTS BENCHMARK (n=28/29, repeats=3)
REST HTTP Latency: p50=838ms, p95=1254ms, retries=0
SSE Streaming TTFB: p50=221ms, p95=352ms (total stream p50=796ms)
RTF: p50=0.1693, p95=0.2204  |  25ms Silence Ratio: 0.240
Round-trip Prisma WER (vs raw text): verbatim=27.39%, rule-normalised=18.83%
Category                 | p50 Lat | p95 Lat | p50 RTF |   Sil | RT RuleWER
numbers                  |   788ms |  1034ms |  0.1763 | 0.232 |       3.6%
abbreviations            |   907ms |   918ms |  0.1881 | 0.230 |      12.5%
code_switching           |   760ms |   848ms |  0.1970 | 0.252 |      50.2%
long_form                |  1458ms |  1773ms |  0.0719 | 0.242 |      13.5%
tricky_pronunciation     |   644ms |   725ms |  0.1501 | 0.175 |       6.8%
punctuation_and_prosody  |   894ms |   991ms |  0.1610 | 0.334 |       3.3%
special_formats          |  1082ms |  1309ms |  0.1687 | 0.228 |      26.3%
regional_languages       |   828ms |   960ms |  0.1692 | 0.238 |      35.5%
edge_cases           

## Live Interactive Check (Native Hinglish Voice + SSE TTFB)

In [6]:
client = GnaniClient()
test_text = 'Aapka OTP 4523 hai, please isse kisi ke saath share mat kijiye.'

sse_out = client.synthesize_stream(text=test_text, language='hi-en')
print(f"SSE voice={sse_out['voice']}: TTFB={sse_out['ttfb_first_chunk_ms']} ms, total={sse_out['total_stream_ms']} ms ({sse_out['num_chunks']} chunks)")

tmp_wav = Path('data/tts/output/demo_sample.wav')
tmp_wav.write_bytes(sse_out['audio_bytes'])
asr_out = client.transcribe(str(tmp_wav), language='hi-IN')
print(f"Prisma hi-IN transcribed in {asr_out['latency_ms']} ms: {asr_out['transcript']}")

SSE voice=Poorvi: TTFB=247.18 ms, total=666.01 ms (154 chunks)
Prisma hi-IN transcribed in 512.69 ms: आपका ओटीपी 4523 है प्लीज इसे किसी के साथ शेयर करें।
